In [1]:
!pip install unsloth[colab-new]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 24.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 868.6/868.6 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 73.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/2

In [2]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

from trl import SFTTrainer, SFTConfig

import torch
from datasets import load_dataset


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer, SFTConfig
import torch
from datasets import load_dataset

In [4]:
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
    max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
    print(f"GPU Detected: {gpu_stats.name}. Max Memory = {max_memory} GB.")
else:
    print("WARNING: GPU Not Detected.")

GPU Detected: Tesla T4. Max Memory = 14.563 GB.


In [5]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    load_in_4bit = True,
    load_in_8bit = False,
    full_finetuning = False,
    dtype = None
)

==((====))==  Unsloth 2026.5.4: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.53G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-1.5b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


In [6]:
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5",
)

In [7]:
model = FastLanguageModel.get_peft_model(
    model,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing="unsloth"
)

Unsloth 2026.5.4 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [8]:
dataset = load_dataset("Ichsan2895/alpaca-gpt4-indonesian", split = "train")
print(f"Jumlah data training: {len(dataset)}")

README.md: 0.00B [00:00, ?B/s]

alpaca-gpt4-indonesia.csv:   0%|          | 0.00/41.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/49969 [00:00<?, ? examples/s]

Jumlah data training: 49969


In [9]:
system_prompt = "Kamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan konteks yang diberikan."


def formatting_prompts_func(examples):
    inputs = examples["input"]
    outputs = examples["output"]

    texts = []

    for input_text, output_text in zip(inputs, outputs):

        conversation = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": input_text},
            {"role": "assistant", "content": output_text},
        ]

        text = tokenizer.apply_chat_template(
            conversation,
            tokenize = False,
            add_generation_prompt = False
        )

        texts.append(text)

    return {"text": texts}


dataset = dataset.map(
    formatting_prompts_func,
    batched = True,
)

Map:   0%|          | 0/49969 [00:00<?, ? examples/s]

In [10]:
dataset[100]["text"]

'<|im_start|>system\nKamu adalah asisten AI yang membantu menjawab pertanyaan pengguna berdasarkan konteks yang diberikan.<|im_end|>\n<|im_start|>user\nTulis cerita pendek tentang seorang orang tua yang memulai kebun.\n<|im_end|>\n<|im_start|>assistant\nDahulu kala, ada seorang wanita tua bernama Martha. Dia tinggal sendirian di sebuah rumah kecil dengan halaman belakang yang telah terbengkalai selama bertahun-tahun. Suatu hari, saat dia duduk di dekat jendela, dia melihat bagaimana rumah-rumah lain di jalanannya memiliki kebun yang indah, penuh dengan bunga dan sayuran. Martha merasa rindu untuk memiliki kebun sendiri, tetapi dia tidak tahu harus mulai dari mana.\n\nTanpa menyerah, Martha mengunjungi toko kebun lokal untuk membuat kebunnya menjadi subur. Sang pemilik, seorang pemuda yang baik hati, menjawab semua pertanyaannya dan membantunya memilih bunga dan sayuran yang tepat untuk kebunnya. Martha pulang dari toko itu dengan perasaan bersemangat dan siap memulai proyek barunya.\n\

In [11]:
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"].select(range(10000))
eval_dataset = dataset["test"].select(range(1000))


In [12]:
sft_config = SFTConfig(
    output_dir="outputs_unsloth",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,

    gradient_checkpointing=True,

    warmup_steps=5,
    max_steps=800,

    learning_rate=2e-4,
    lr_scheduler_type="linear",

    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),

    logging_steps=10,

    eval_strategy="steps",
    eval_steps=100,

    save_steps=100,

    optim="paged_adamw_8bit",

    dataset_text_field="text",

    max_length=256,
)

In [13]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    learning_rate = 2e-4,
    args=sft_config,
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10000 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [14]:
trainer.model.print_trainable_parameters()

trainable params: 18,464,768 || all params: 1,562,179,072 || trainable%: 1.1820


In [15]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,000 | Num Epochs = 1 | Total steps = 800
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 4 x 1) = 4
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
100,1.329257,1.232560
200,1.210835,1.216245
300,1.262780,1.205894
400,1.233677,1.198701
500,1.193511,1.192559
600,1.145868,1.186755
700,1.203253,1.182803
800,1.246450,1.181362


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Unsloth: Restored added_tokens_decoder metadata in outputs_unsloth/checkpoint-100/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_unsloth/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_unsloth/checkpoint-300/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_unsloth/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_unsloth/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_unsloth/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_unsloth/checkpoint-700/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs_unsloth/checkpoint-800/tokenizer_config.json.


TrainOutput(global_step=800, training_loss=1.213829197883606, metrics={'train_runtime': 2199.388, 'train_samples_per_second': 1.455, 'train_steps_per_second': 0.364, 'total_flos': 7126962590631936.0, 'train_loss': 1.213829197883606, 'epoch': 0.32})

In [16]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151665)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

In [17]:
from transformers import TextStreamer

FastLanguageModel.for_inference(model)

messages = [
    {
        "role": "system",
        "content": "Anda adalah asisten cerdas yang menjawab pertanyaan secara akurat berdasarkan fakta."
    },
    {
        "role": "user",
        "content": "Sebutkan 3 tempat wisata tercantik di Indonesia beserta lokasinya!"
    }
]

inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")

text_streamer = TextStreamer(
    tokenizer,
    skip_prompt=True
)

_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=256,
    temperature=0.3,
    repetition_penalty=1.2,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Both `max_new_tokens` (=256) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_

1) Pantai Canggu, Bali: Pantai ini dikenal dengan keindahan pantainya dan pemandangan alamnya yang indah.
2) Taman Nasional Gunung Bromo-Tengger-Semeru (Gunung Semeru): Ini merupakan gunung tertinggi di Asia Timur dan salah satu destinasi wisata favorit bagi para penjelajahi dunia.
3) Pulau Komodo: Di sebelah barat laut Nusa Tenggara Barat ada pulau komodo yang populer untuk pendaki dan pengunjung lainnya karena ekosistemnya yang unik dan banyak spesies burung liar seperti kuda putih dan serigala.<|im_end|>


In [18]:
model.save_pretrained("fine_tuned_qwen")
tokenizer.save_pretrained("fine_tuned_qwen")

Unsloth: Restored added_tokens_decoder metadata in fine_tuned_qwen/tokenizer_config.json.


('fine_tuned_qwen/tokenizer_config.json',
 'fine_tuned_qwen/chat_template.jinja',
 'fine_tuned_qwen/tokenizer.json')

In [19]:
from huggingface_hub import login
login()

In [21]:
model.push_to_hub_merged(
    "rafiqiraihan/qwen-rag-indonesia",
    tokenizer,
    save_method = "merged_16bit",
)

No files have been modified since last commit. Skipping to prevent empty commit.
Unsloth: Restored added_tokens_decoder metadata in rafiqiraihan/qwen-rag-indonesia/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...-indonesia/tokenizer.json: 100%|##########| 11.4MB / 11.4MB            

Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...donesia/model.safetensors:   0%|          | 7.93MB / 3.09GB            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [01:38<00:00, 98.05s/it]


Unsloth: Merge process complete. Saved to `/content/rafiqiraihan/qwen-rag-indonesia`
